# Laboratorio Lakehouse · Big Data Unidad 4
## Caso Lácteos del Altiplano S.A.S. (empresa ficticia)

**Objetivo:** construir un Lakehouse con arquitectura medallón (landing → bronze → silver → gold) sobre un almacenamiento de objetos compatible con S3, usando tablas **Delta Lake**, y responder la pregunta de la gerencia:

> ¿Cuánto vendimos de lotes cuya cadena de frío se rompió (más de 6 °C)?

**Cómo trabajar:** ejecuten las celdas **en orden**, una por una, con **Shift + Enter**. Lean el texto antes de cada celda y la sección *Qué observar* después de ejecutarla. No se instala nada en el computador: todo corre en la nube de Google Colab.

## Paso 0 · Instalar las librerías (≈ 1 minuto)
| Librería | Papel en la arquitectura |
|---|---|
| `deltalake` | Formato de tabla **Delta Lake** (crea el `_delta_log`) |
| `duckdb` | Motor de consultas **SQL** |
| `moto[server]` | Servidor **S3 local** (simula AWS S3 / MinIO) |
| `boto3` | Cliente oficial para hablar con **S3** |

In [ ]:
%pip install -q deltalake duckdb "moto[server]" boto3
print("✅ Librerías instaladas")

## Paso 1 · Encender el almacenamiento de objetos S3 y crear el *bucket*
En una empresa, este almacenamiento sería **AWS S3**, **Azure Blob**, **Google Cloud Storage** o **MinIO**. Aquí encendemos un servidor S3 dentro del propio Colab. **El código del resto del laboratorio sería idéntico en la vida real: solo cambian la dirección (endpoint) y las credenciales.**

In [ ]:
import subprocess, sys, time, os, json
import boto3

ENDPOINT = "http://127.0.0.1:5055"
servidor = subprocess.Popen([sys.executable, "-m", "moto.server", "-p", "5055"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

s3 = boto3.client("s3", endpoint_url=ENDPOINT, region_name="us-east-1",
                  aws_access_key_id="admin", aws_secret_access_key="admin123")
for _ in range(60):                      # espera a que el servidor responda
    try:
        s3.list_buckets(); break
    except Exception:
        time.sleep(0.5)

BUCKET = "lakehouse"
if BUCKET not in [b["Name"] for b in s3.list_buckets().get("Buckets", [])]:
    s3.create_bucket(Bucket=BUCKET)

# Opciones que Delta Lake necesita para hablar con un S3 que no es AWS
OPTS = {"AWS_ENDPOINT_URL": ENDPOINT, "AWS_ACCESS_KEY_ID": "admin",
        "AWS_SECRET_ACCESS_KEY": "admin123", "AWS_REGION": "us-east-1",
        "AWS_ALLOW_HTTP": "true", "AWS_S3_ALLOW_UNSAFE_RENAME": "true"}
def uri(ruta):
    return f"s3://{BUCKET}/{ruta}"

def listar(prefijo=""):
    for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=prefijo).get("Contents", []):
        print(f'{o["Key"]:70s} {o["Size"]:>8,} bytes')

print("✅ S3 encendido. Buckets:", [b["Name"] for b in s3.list_buckets()["Buckets"]])

**Qué observar:** aparece el bucket `lakehouse`. Un *bucket* es el contenedor principal del almacenamiento de objetos; allí vivirán **todas** las capas del Lakehouse.

## Paso 2 · Generar los datos de la empresa
Se crean tres archivos sintéticos (siempre los mismos, gracias a una semilla fija):
- `ventas_erp.csv` → 302 ventas del ERP (**estructurado**), con errores escondidos a propósito.
- `sensores_tanques.jsonl` → 240 lecturas de temperatura de los tanques (**semiestructurado**).
- `correcciones_erp.csv` → 5 ventas corregidas y 5 ventas nuevas.

In [ ]:
import csv, random
from datetime import datetime, timedelta
from pathlib import Path
random.seed(2026)
Path("datos").mkdir(exist_ok=True)
SUC = ["Tunja", "Duitama", "Sogamoso", "Chiquinquirá"]
PROD = {"Leche entera 1L": 3900, "Yogur fresa 1L": 7800, "Queso campesino 500g": 12500, "Kumis 1L": 6900}
LOTES = [f"L{n:03d}" for n in range(1, 21)]
inicio = datetime(2026, 9, 1)

filas = []
for i in range(1, 301):
    p = random.choice(list(PROD))
    filas.append({"id_venta": f"V{i:05d}", "fecha": (inicio + timedelta(days=random.randint(0, 29))).strftime("%Y-%m-%d"),
                  "sucursal": random.choice(SUC), "producto": p, "id_lote": random.choice(LOTES),
                  "cantidad": random.randint(1, 40), "precio_unitario": PROD[p]})
filas[10]["cantidad"] = ""        # error: cantidad vacía
filas[20]["cantidad"] = -5        # error: cantidad negativa
filas.append(dict(filas[30])); filas.append(dict(filas[31]))   # error: duplicados
with open("datos/ventas_erp.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(filas[0])); w.writeheader(); w.writerows(filas)

with open("datos/sensores_tanques.jsonl", "w", encoding="utf-8") as f:
    for lote in LOTES:
        base = inicio + timedelta(days=random.randint(0, 25)); tq = random.choice(["TQ-01", "TQ-02", "TQ-03", "TQ-04"])
        roto = lote in ("L004", "L013", "L017")
        for h in range(12):
            t = round(random.uniform(2.5, 4.2), 2)
            if roto and h in (5, 6, 7): t = round(random.uniform(7.5, 10.5), 2)
            r = {"id_tanque": tq, "id_lote": lote, "ts": (base + timedelta(hours=h)).isoformat(), "temperatura_c": t,
                 "ph": round(random.uniform(6.5, 6.8), 2),
                 "ubicacion": {"vereda": random.choice(["Runta", "La Hoya", "Tras del Alto"]), "municipio": "Tunja"}}
            if h % 4 == 0: r["firmware"] = "v2.1"
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

corr = []
for i in (1, 2, 3, 4, 5):
    r = dict(filas[i - 1]); r["cantidad"] = int(r["cantidad"]) + 100; corr.append(r)
for j in range(301, 306):
    corr.append({"id_venta": f"V{j:05d}", "fecha": "2026-09-30", "sucursal": "Tunja", "producto": "Kumis 1L",
                 "id_lote": "L020", "cantidad": 12, "precio_unitario": 6900})
with open("datos/correcciones_erp.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(corr[0])); w.writeheader(); w.writerows(corr)
print("✅ Datos generados:", sorted(os.listdir("datos")))

## Paso 3 · Capa LANDING: subir los archivos tal como llegan
La capa *landing* es la zona de aterrizaje: los archivos se guardan **sin modificarlos**.

In [ ]:
for nombre in ["ventas_erp.csv", "sensores_tanques.jsonl", "correcciones_erp.csv"]:
    s3.upload_file(f"datos/{nombre}", BUCKET, f"landing/{nombre}")
listar("landing/")

**Qué observar:** tres objetos con el prefijo `landing/`. En S3 **no existen carpetas reales**: `landing/` es solo parte del nombre (la *clave*) del objeto.

## Paso 4 · Capa BRONZE: convertir lo crudo en tablas Delta
Bronze guarda **los mismos datos**, pero ya como tablas Delta Lake (con historial). Todavía **no se limpian**: las ventas se leen todas como texto.

In [ ]:
import pandas as pd
from deltalake import DeltaTable, write_deltalake

ventas_raw = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="landing/ventas_erp.csv")["Body"], dtype=str)
write_deltalake(uri("bronze/ventas"), ventas_raw, mode="overwrite", schema_mode="overwrite", storage_options=OPTS)

lineas = s3.get_object(Bucket=BUCKET, Key="landing/sensores_tanques.jsonl")["Body"].read().decode().splitlines()
sensores_raw = pd.json_normalize([json.loads(l) for l in lineas])
write_deltalake(uri("bronze/sensores"), sensores_raw, mode="overwrite", schema_mode="overwrite", storage_options=OPTS)

print("bronze/ventas  :", len(ventas_raw), "filas")
print("bronze/sensores:", len(sensores_raw), "filas")
print("Columnas de sensores:", list(sensores_raw.columns))
ventas_raw.head()

**Qué observar:** el JSON anidado (`ubicacion`) se aplanó en dos columnas. La columna `firmware` aparece aunque solo algunas lecturas la traen: así es un dato **semiestructurado**.

## Paso 5 · Por dentro de una tabla Delta: Parquet + `_delta_log`
Una tabla Delta es una carpeta con archivos **Parquet** (los datos) y un **diario** (`_delta_log`) que registra cada cambio.

In [ ]:
listar("bronze/ventas/")
print()
diario = s3.get_object(Bucket=BUCKET, Key="bronze/ventas/_delta_log/00000000000000000000.json")["Body"].read().decode()
for linea in diario.splitlines():
    accion = json.loads(linea)
    tipo = list(accion)[0]
    detalle = accion[tipo].get("path", "") if tipo == "add" else ""
    print(f"Acción del diario: {tipo:11s} {detalle}")

**Qué observar:** el archivo `00000000000000000000.json` es la **versión 0** de la tabla. Su acción `add` dice qué archivo Parquet forma parte de la tabla. **Si un Parquet no está registrado en el diario, para Delta no existe.** Esto es lo que da las transacciones ACID.

## Paso 6 · Capa SILVER: limpiar y dar tipos correctos
Reglas de calidad: eliminar duplicados, descartar cantidades vacías o negativas, convertir fechas y números, y calcular el total.

In [ ]:
v = DeltaTable(uri("bronze/ventas"), storage_options=OPTS).to_pandas()
antes = len(v)
v = v.drop_duplicates()
v["cantidad"] = pd.to_numeric(v["cantidad"], errors="coerce")
v = v[v["cantidad"].notna() & (v["cantidad"] > 0)].copy()
v["cantidad"] = v["cantidad"].astype("int64")
v["precio_unitario"] = v["precio_unitario"].astype("int64")
v["fecha"] = pd.to_datetime(v["fecha"]).dt.date
v["total"] = v["cantidad"] * v["precio_unitario"]
v = v.reset_index(drop=True)
write_deltalake(uri("silver/ventas"), v, mode="overwrite", schema_mode="overwrite", storage_options=OPTS)

s = DeltaTable(uri("bronze/sensores"), storage_options=OPTS).to_pandas()
s["ts"] = pd.to_datetime(s["ts"])
s = s.rename(columns={"ubicacion.vereda": "vereda", "ubicacion.municipio": "municipio"})
write_deltalake(uri("silver/sensores"), s, mode="overwrite", schema_mode="overwrite", storage_options=OPTS)

print(f"silver/ventas: {antes} → {len(v)} filas (se eliminaron {antes - len(v)} filas con errores)")
print("silver/sensores:", len(s), "filas")

**Qué observar:** de 302 ventas quedan **298**. ¿Cuáles cuatro filas se eliminaron y por qué? (Pista: revisen el Paso 2).

## Paso 7 · MERGE: aplicar las correcciones del ERP (ACID)
Llegan 5 ventas corregidas y 5 ventas nuevas. **MERGE** actualiza las existentes e inserta las nuevas en **una sola transacción**.

In [ ]:
corr = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="landing/correcciones_erp.csv")["Body"])
corr["fecha"] = pd.to_datetime(corr["fecha"]).dt.date
corr["total"] = corr["cantidad"] * corr["precio_unitario"]

resultado = (DeltaTable(uri("silver/ventas"), storage_options=OPTS)
             .merge(source=corr, predicate="t.id_venta = s.id_venta", source_alias="s", target_alias="t")
             .when_matched_update_all()
             .when_not_matched_insert_all()
             .execute())
print("Filas actualizadas:", resultado["num_target_rows_updated"])
print("Filas insertadas  :", resultado["num_target_rows_inserted"])
print()
listar("silver/ventas/_delta_log/")

**Qué observar:** 5 actualizadas y 5 insertadas. En el diario apareció un **segundo** archivo JSON: la versión 1. Los Parquet nunca se modifican; se escribe uno nuevo y el diario marca el viejo como reemplazado.

## Paso 8 · Time travel: consultar el pasado

In [ ]:
dt = DeltaTable(uri("silver/ventas"), storage_options=OPTS)
for h in dt.history():
    print("Versión", h["version"], "→", h["operation"])

v0 = DeltaTable(uri("silver/ventas"), version=0, storage_options=OPTS).to_pandas()
v1 = dt.to_pandas()
print()
print("Venta V00001 en la versión 0:", v0.loc[v0.id_venta == "V00001", "cantidad"].item(), "unidades")
print("Venta V00001 hoy            :", v1.loc[v1.id_venta == "V00001", "cantidad"].item(), "unidades")
print("Filas en la versión 0:", len(v0), "| filas hoy:", len(v1))

**Qué observar:** la venta V00001 tenía **39** unidades y hoy tiene **139**. Las dos versiones siguen disponibles. Así se auditaría «cómo estaban los datos el día del reporte».

## Paso 9 · Control de esquema: el guardián de la tabla
Llega una venta con una columna que la tabla no conoce (`canal`). Primero Delta la **rechaza**; luego la aceptamos **autorizando** la evolución del esquema.

In [ ]:
nueva = corr.head(1).copy()
nueva["id_venta"] = "V09999"
nueva["canal"] = "tienda en línea"

try:
    write_deltalake(uri("silver/ventas"), nueva, mode="append", storage_options=OPTS)
except Exception as e:
    print("❌ RECHAZADA por control de esquema →", type(e).__name__)

write_deltalake(uri("silver/ventas"), nueva, mode="append", schema_mode="merge", storage_options=OPTS)
print("✅ Aceptada al autorizar la evolución. Columnas ahora:")
print([c.name for c in DeltaTable(uri("silver/ventas"), storage_options=OPTS).schema().fields])

**Qué observar:** en un Data Lake sin Delta, el archivo se habría guardado sin aviso y los lectores fallarían después. Aquí la tabla se protege.

## Paso 10 · Capa GOLD: responder a la gerencia con SQL
Se unen **ventas** (origen ERP) y **sensores** (origen IoT) por `id_lote`. Un lote está **EN RIESGO** si alguna lectura superó 6 °C.

In [ ]:
import duckdb
ventas = DeltaTable(uri("silver/ventas"), storage_options=OPTS).to_pyarrow_table()
sens = DeltaTable(uri("silver/sensores"), storage_options=OPTS).to_pyarrow_table()

gold = duckdb.sql('''
    WITH calidad AS (
        SELECT id_lote,
               MAX(temperatura_c) AS temp_max,
               COUNT(*) FILTER (WHERE temperatura_c > 6)::INT AS lecturas_fuera_rango
        FROM sens GROUP BY id_lote)
    SELECT v.id_lote,
           ROUND(c.temp_max, 2) AS temp_max_c,
           c.lecturas_fuera_rango,
           CASE WHEN c.lecturas_fuera_rango > 0 THEN 'EN RIESGO' ELSE 'OK' END AS estado_frio,
           SUM(v.cantidad)::INT AS unidades,
           SUM(v.total)::BIGINT AS ingresos_cop
    FROM ventas v JOIN calidad c USING (id_lote)
    GROUP BY ALL
    ORDER BY estado_frio, ingresos_cop DESC
''').df()
write_deltalake(uri("gold/kpi_cadena_frio"), gold, mode="overwrite", schema_mode="overwrite", storage_options=OPTS)

riesgo = gold[gold.estado_frio == "EN RIESGO"]
print("Lotes en riesgo:", ", ".join(riesgo.id_lote))
print(f"💰 RESPUESTA A LA GERENCIA: ${riesgo.ingresos_cop.sum():,.0f} COP en ventas de lotes con la cadena de frío rota")
gold.head(6)

**Qué observar:** tres lotes en riesgo (**L004, L013, L017**). BI y ciencia de datos leen **esta misma tabla gold**: ya no hay dos cifras.

## Paso 11 · Inventario final del Lakehouse

In [ ]:
for capa in ["landing/", "bronze/", "silver/", "gold/"]:
    objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=capa).get("Contents", [])
    print(f"{capa:9s} {len(objs):3d} objetos   {sum(o['Size'] for o in objs):>9,} bytes")

**Qué observar:** las cuatro capas viven en **un solo bucket**. La arquitectura medallón es una organización lógica sobre el mismo almacenamiento físico.

## 🏆 Reto del equipo (10 minutos)
Creen la tabla gold **`gold/ingresos_sucursal`** con los ingresos totales por sucursal, ordenados de mayor a menor, usando la tabla `ventas` de silver. Escriban el SQL en la celda siguiente.

In [ ]:
reto = duckdb.sql('''
    -- Escriban aquí su consulta SQL
    SELECT 'completar' AS sucursal
''').df()
reto

## Así sería en una empresa real
| En este laboratorio | En producción |
|---|---|
| Servidor S3 dentro de Colab (`moto`) | AWS S3, Azure Blob, Google Cloud Storage o MinIO |
| Credenciales `admin` / `admin123` | Usuarios con permisos (IAM), cifrado y HTTPS |
| `pandas` + `deltalake` | Spark o Databricks sobre muchos servidores |
| `duckdb` | Databricks SQL, Trino o Spark SQL |
| Ejecutar celdas a mano | Pipelines programados con Airflow o Databricks Workflows |
| Un solo usuario escribiendo | Muchos escritores coordinados por el formato Delta |

**Los archivos Delta que crearon (Parquet + `_delta_log`) son exactamente los mismos que se usan en la industria.**